# CSET419 – Introduction to Generative AI
## Lab 9 – Generative Models for Sequential Data

---
## Component I: Sequence Generation using LSTM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Dataset ──────────────────────────────────────────────────────────────────
corpus = """machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.
long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.
language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.
generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance."""

# ── Preprocessing ─────────────────────────────────────────────────────────────
words = corpus.lower().replace(".", "").split()
vocab = sorted(set(words))
w2i   = {w: i for i, w in enumerate(vocab)}
i2w   = {i: w for w, i in w2i.items()}
V     = len(vocab)

print(f"Vocabulary size : {V}")
print(f"Total words     : {len(words)}")

In [ ]:
# ── Input-Output Sequence Pairs ───────────────────────────────────────────────
SEQ_LEN = 3
X, Y = [], []
for i in range(len(words) - SEQ_LEN):
    X.append([w2i[w] for w in words[i:i+SEQ_LEN]])
    Y.append(w2i[words[i+SEQ_LEN]])
X, Y = np.array(X), np.array(Y)

print(f"Training samples: {len(X)}")
print(f"Example -> Input: {words[:SEQ_LEN]}  Target: '{words[SEQ_LEN]}'")

In [ ]:
# ── LSTM Model (from scratch with NumPy) ──────────────────────────────────────
np.random.seed(42)
EMBED, HIDDEN, LR, EPOCHS = 16, 32, 0.05, 300
I = EMBED + HIDDEN

def xavier(r, c): return np.random.randn(r, c) * np.sqrt(2/(r+c))

E  = np.random.randn(V, EMBED) * 0.1
Wf = xavier(HIDDEN,I); bf = np.zeros(HIDDEN)
Wi = xavier(HIDDEN,I); bi = np.zeros(HIDDEN)
Wo = xavier(HIDDEN,I); bo = np.zeros(HIDDEN)
Wg = xavier(HIDDEN,I); bg = np.zeros(HIDDEN)
Wy = xavier(V,HIDDEN); by = np.zeros(V)

def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-15,15)))
def softmax(x): e=np.exp(x-x.max()); return e/e.sum()

def forward(seq_idx):
    h, c = np.zeros(HIDDEN), np.zeros(HIDDEN)
    cache = []
    for idx in seq_idx:
        x = E[idx]
        z = np.concatenate([x, h])
        f = sigmoid(Wf@z+bf)
        i = sigmoid(Wi@z+bi)
        o = sigmoid(Wo@z+bo)
        g = np.tanh(Wg@z+bg)
        c = f*c + i*g
        h = o*np.tanh(c)
        cache.append((z,f,i,o,g,c,h))
    probs = softmax(Wy@h+by)
    return probs, h, cache

# ── Training ──────────────────────────────────────────────────────────────────
losses = []
for epoch in range(EPOCHS):
    idx_list = np.random.permutation(len(X))
    epoch_loss = 0
    for idx in idx_list[:40]:
        seq, target = X[idx], Y[idx]
        probs, h, cache = forward(seq)
        epoch_loss += -np.log(probs[target]+1e-9)

        dlogits = probs.copy(); dlogits[target] -= 1
        dWy = np.outer(dlogits, h); dby = dlogits
        dh  = Wy.T @ dlogits

        grads = {k: np.zeros_like(v) for k,v in
                 [("Wf",Wf),("Wi",Wi),("Wo",Wo),("Wg",Wg),
                  ("bf",bf),("bi",bi),("bo",bo),("bg",bg)]}
        dh_next = dh; dc_next = np.zeros(HIDDEN)
        for t in reversed(range(len(seq))):
            z,f,i,o,g,c_t,h_t = cache[t]
            c_prev = cache[t-1][5] if t>0 else np.zeros(HIDDEN)
            tc = np.tanh(c_t)
            do = dh_next*tc
            dc = dh_next*o*(1-tc**2)+dc_next
            df = dc*c_prev; di = dc*g; dg = dc*i; dc_next = dc*f
            dsf=sigmoid(Wf@z+bf)*(1-sigmoid(Wf@z+bf))*df
            dsi=sigmoid(Wi@z+bi)*(1-sigmoid(Wi@z+bi))*di
            dso=sigmoid(Wo@z+bo)*(1-sigmoid(Wo@z+bo))*do
            dsg=(1-g**2)*dg
            grads["Wf"]+=np.outer(dsf,z); grads["bf"]+=dsf
            grads["Wi"]+=np.outer(dsi,z); grads["bi"]+=dsi
            grads["Wo"]+=np.outer(dso,z); grads["bo"]+=dso
            grads["Wg"]+=np.outer(dsg,z); grads["bg"]+=dsg
            dh_next = np.zeros(HIDDEN)

        clip=1.0
        Wf-=LR*np.clip(grads["Wf"],-clip,clip)
        Wi-=LR*np.clip(grads["Wi"],-clip,clip)
        Wo-=LR*np.clip(grads["Wo"],-clip,clip)
        Wg-=LR*np.clip(grads["Wg"],-clip,clip)
        bf-=LR*np.clip(grads["bf"],-clip,clip)
        bi-=LR*np.clip(grads["bi"],-clip,clip)
        bo-=LR*np.clip(grads["bo"],-clip,clip)
        bg-=LR*np.clip(grads["bg"],-clip,clip)
        Wy-=LR*np.clip(dWy,-clip,clip)
        by-=LR*np.clip(dby,-clip,clip)

    avg = epoch_loss/40
    losses.append(avg)
    if (epoch+1)%50==0:
        print(f"Epoch {epoch+1:3d} | Loss: {avg:.4f}")

print(f"\nFinal Loss: {losses[-1]:.4f}")

plt.figure(figsize=(8,4))
plt.plot(losses, color='steelblue', linewidth=2)
plt.title('LSTM Training Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── Generate Sequences ────────────────────────────────────────────────────────
def generate_lstm(seed_words, n=8, temperature=0.9):
    seq = [w2i.get(w,0) for w in seed_words[-SEQ_LEN:]]
    result = list(seed_words)
    for _ in range(n):
        probs,_,_ = forward(seq[-SEQ_LEN:])
        probs = np.log(probs+1e-9)/temperature
        probs = np.exp(probs-probs.max()); probs/=probs.sum()
        nxt = np.random.choice(V, p=probs)
        result.append(i2w[nxt]); seq.append(nxt)
    return " ".join(result)

print("=" * 55)
print("       LSTM – GENERATED SEQUENCES (Expected Output)")
print("=" * 55)
for seed in [["machine","learning","models"],
             ["lstm","uses","gates"],
             ["sequence","models","process"],
             ["deep","learning","improves"],
             ["language","modeling","predicts"]]:
    print(f"\nSeed : '{' '.join(seed)}'")
    print(f"  ->  {generate_lstm(seed)}")

---
## Component II: Transformer-Based Sequence Generation

In [ ]:
# ── Input-Output Pairs (seq_len=4 for Transformer) ───────────────────────────
SEQ_LEN_T = 4
X_t, Y_t = [], []
for i in range(len(words)-SEQ_LEN_T):
    X_t.append([w2i[w] for w in words[i:i+SEQ_LEN_T]])
    Y_t.append(w2i[words[i+SEQ_LEN_T]])
X_t, Y_t = np.array(X_t), np.array(Y_t)
print(f"Training samples: {len(X_t)}")

In [ ]:
# ── Positional Encoding ───────────────────────────────────────────────────────
def positional_encoding(seq_len, d_model):
    pe  = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:,None]
    div = np.exp(np.arange(0,d_model,2)*(-np.log(10000)/d_model))
    pe[:,0::2] = np.sin(pos*div)
    pe[:,1::2] = np.cos(pos*div[:d_model//2])
    return pe

D_MODEL = 32
PE = positional_encoding(SEQ_LEN_T, D_MODEL)

plt.figure(figsize=(8,3))
plt.imshow(PE.T, aspect='auto', cmap='RdBu')
plt.colorbar(); plt.title('Positional Encoding')
plt.xlabel('Position'); plt.ylabel('Dimension')
plt.tight_layout(); plt.show()
print(f"PE shape: {PE.shape}")

In [ ]:
# ── Transformer Model (from scratch with NumPy) ───────────────────────────────
np.random.seed(7)
D_FF = 64; LR_T = 0.05; EPOCHS_T = 400

E_t  = np.random.randn(V, D_MODEL)*0.1
Wq   = xavier(D_MODEL,D_MODEL); Wk = xavier(D_MODEL,D_MODEL)
Wv   = xavier(D_MODEL,D_MODEL); Wo_t= xavier(D_MODEL,D_MODEL)
W1   = xavier(D_FF,D_MODEL);    b1  = np.zeros(D_FF)
W2   = xavier(D_MODEL,D_FF);    b2  = np.zeros(D_MODEL)
ln1_g= np.ones(D_MODEL);        ln1_b=np.zeros(D_MODEL)
ln2_g= np.ones(D_MODEL);        ln2_b=np.zeros(D_MODEL)
Wout = xavier(V,D_MODEL);       bout= np.zeros(V)

def layer_norm(x,g,b,eps=1e-6):
    mu=x.mean(-1,keepdims=True); sig=x.std(-1,keepdims=True)+eps
    return g*(x-mu)/sig+b

def softmax2d(x):
    e=np.exp(x-x.max(axis=-1,keepdims=True)); return e/e.sum(axis=-1,keepdims=True)

def relu(x): return np.maximum(0,x)

def forward_transformer(seq_idx):
    T   = len(seq_idx)
    x   = E_t[seq_idx] + PE[:T]
    Q,K,Vm = x@Wq.T, x@Wk.T, x@Wv.T
    scores = Q@K.T/np.sqrt(D_MODEL) + np.triu(np.full((T,T),-1e9),1)
    attn   = softmax2d(scores)
    x1 = layer_norm(x + attn@Vm@Wo_t.T, ln1_g, ln1_b)
    ff = relu(x1@W1.T+b1)@W2.T+b2
    x2 = layer_norm(x1+ff, ln2_g, ln2_b)
    probs = softmax(Wout@x2[-1]+bout)
    return probs, x2[-1], attn, x

# ── Training ──────────────────────────────────────────────────────────────────
losses_t = []
for epoch in range(EPOCHS_T):
    idx_list = np.random.permutation(len(X_t))
    epoch_loss = 0
    for idx in idx_list[:50]:
        seq, target = X_t[idx].tolist(), Y_t[idx]
        probs, last, attn_w, x_emb = forward_transformer(seq)
        epoch_loss += -np.log(probs[target]+1e-9)

        dlogits = probs.copy(); dlogits[target] -= 1
        dWout   = np.outer(dlogits, last)
        dbout   = dlogits
        dlast   = Wout.T@dlogits
        dlast_n = dlast*ln2_g
        dW2     = np.outer(dlast_n, relu(last@W1.T+b1))
        db2     = dlast_n
        dWo_t   = np.outer(dlast_n, attn_w[-1]@(x_emb@Wv.T))
        dE_update = Wout.T@dlogits

        clip=1.0
        Wout -=LR_T*np.clip(dWout,-clip,clip)
        bout -=LR_T*np.clip(dbout,-clip,clip)
        W2   -=LR_T*np.clip(dW2,  -clip,clip)
        b2   -=LR_T*np.clip(db2,  -clip,clip)
        Wo_t -=LR_T*np.clip(dWo_t,-clip,clip)
        for tok in seq:
            E_t[tok]-=LR_T*0.01*np.clip(dE_update,-clip,clip)

    avg=epoch_loss/50; losses_t.append(avg)
    if (epoch+1)%100==0:
        print(f"Epoch {epoch+1:3d} | Loss: {avg:.4f}")

print(f"\nFinal Loss: {losses_t[-1]:.4f}")

plt.figure(figsize=(8,4))
plt.plot(losses_t, color='darkorange', linewidth=2)
plt.title('Transformer Training Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── Generate Sequences ────────────────────────────────────────────────────────
def generate_transformer(seed_words, n=8, temperature=0.85):
    seq = [w2i.get(w,0) for w in seed_words][-SEQ_LEN_T:]
    result = list(seed_words)
    for _ in range(n):
        probs,_,_,_ = forward_transformer(seq[-SEQ_LEN_T:])
        p = np.log(probs+1e-9)/temperature
        p = np.exp(p-p.max()); p/=p.sum()
        nxt = np.random.choice(V,p=p)
        result.append(i2w[nxt]); seq.append(nxt)
    return " ".join(result)

print("=" * 58)
print("   TRANSFORMER – GENERATED SEQUENCES (Expected Output)")
print("=" * 58)
for seed in [["sequence","generation","is","widely"],
             ["lstm","uses","gates","to"],
             ["deep","learning","improves","sequence"],
             ["language","modeling","predicts","the"],
             ["generative","models","learn","probability"]]:
    print(f"\nSeed : '{' '.join(seed)}'")
    print(f"  ->  {generate_transformer(seed)}")

In [ ]:
# ── Loss Comparison Plot ──────────────────────────────────────────────────────
plt.figure(figsize=(8,4))
plt.plot(losses,   color='steelblue',  linewidth=2, label=f'LSTM (final={losses[-1]:.4f})')
plt.plot(losses_t, color='darkorange', linewidth=2, label=f'Transformer (final={losses_t[-1]:.4f})')
plt.title('Training Loss: LSTM vs Transformer')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()